In [20]:
# imports
import os
import sys
import json
import shutil
from IPython.display import Image, display
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [6]:
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"
src_path = os.path.join(repo_dir, "src")

# Change working directory to the repo root
os.chdir(repo_dir)
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification


In [12]:
# Download dataset zip if missing
if not os.path.exists("imagesoasis.zip"):
    print("Downloading dataset from Kaggle...")
    !kaggle datasets download -d ninadaithal/imagesoasis
else:
    print("imagesoasis.zip already exists locally.")

# Unzip dataset inside content/
print("Unzipping dataset to local Colab SSD...")
os.makedirs("/content/dataset", exist_ok=True)
!unzip -qo imagesoasis.zip -d /content/dataset
print("Unzipped!")

imagesoasis.zip already exists locally.
Unzipping dataset to local Colab SSD...
Unzipped!


In [14]:
'''VERIFY DATASET STRUCTURE & 4 CLASSES'''
# Check subdirectories to ensure all 4 classes (Mild, Moderate, Non, Very mild) exist
local_dataset_path = "/content/dataset/Data"
extracted_subdirs = []
for root, dirs, files in os.walk(local_dataset_path):
  for d in dirs:
    extracted_subdirs.append(d)

print("Detected folders/classes in dataset directory:", set(extracted_subdirs))

Detected folders/classes in dataset directory: {'Moderate Dementia', 'Non Demented', 'Mild Dementia', 'Very mild Dementia'}


In [15]:
'''RUN TRAINING'''
# Locate where the actual class subfolders reside (handling potential nesting)
actual_data_dir = local_dataset_path
for root, dirs, files in os.walk(local_dataset_path):
  if any("Demented" in d for d in dirs):
    actual_data_dir = root
    break

print(f"\n--- STARTING TRAINING (5 EPOCHS) using data at: {actual_data_dir} ---")
!python src/train.py --data_dir "{actual_data_dir}" --epochs 5 --batch_size 32


--- STARTING TRAINING (5 EPOCHS) using data at: /content/dataset/Data ---
Using device: cuda
Split mode: subject
Subjects — total=347, train=242, val=52, test=53
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 105MB/s]
Class weights (train rebalanced): [0.2468000054359436, 3.6405999660491943, 0.019200000911951065, 0.093299999833107]
Epoch 01/5 | train_loss=0.3050 train_acc=0.8784 | val_loss=0.7411 val_acc=0.7971
Epoch 02/5 | train_loss=0.0798 train_acc=0.9678 | val_loss=1.1078 val_acc=0.7381
Epoch 03/5 | train_loss=0.0494 train_acc=0.9810 | val_loss=1.1267 val_acc=0.7997
Epoch 04/5 | train_loss=0.0417 train_acc=0.9843 | val_loss=1.1904 val_acc=0.7616
Epoch 05/5 | train_loss=0.0345 train_acc=0.9858 | val_loss=1.0537 val_acc=0.7886
Best validation accuracy: 0.7997
Saved best model to: outputs/best_model.pt
Test loss: 1.3603 | Test accuracy: 0.7561
Classification report:
  

In [17]:
import os
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"

print("Searching for saved model files (.pt / .pth):")
for root, dirs, files in os.walk(repo_dir):
  for file in files:
    if file.endswith(('.pt', '.pth', '.ckpt')):
      print(os.path.join(root, file))

Searching for saved model files (.pt / .pth):
/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/best_model.pt
/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/model_bundle.pt


In [19]:
'''RUN EVALUATION & GENERATE ARTIFACTS'''
print("\n--- RUNNING EVALUATION ---")
# Run evaluation pointing to best model weights
!python src/evaluate.py --data_dir "/content/dataset/Data" --model_bundle "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/best_model.pt"


--- RUNNING EVALUATION ---
Traceback (most recent call last):
  File "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/src/evaluate.py", line 147, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/src/evaluate.py", line 71, in main
    class_names = bundle["class_names"]
                  ~~~~~~^^^^^^^^^^^^^^^
KeyError: 'class_names'


# evaluation

In [25]:
# Ensure path is set
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"
src_path = os.path.join(repo_dir, "src")
if src_path not in sys.path:
  sys.path.append(src_path)

from dataset import make_stratified_splits

# Load data loaders
print("Loading test data...")
_, _, test_loader, _ = make_stratified_splits(
  data_dir="/content/dataset/Data",
  batch_size=32,
  split_mode="subject"
)
print("Test data loaded successfully!")

Loading test data...
Test data loaded successfully!


In [27]:
# Setup model architecture (ResNet18 with 4 classes matching training)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [28]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4) # 4 dementia classes

# Load best_model.pt weights
model_path = os.path.join(repo_dir, "outputs/best_model.pt")
state_dict = torch.load(model_path, map_location=device)

In [29]:
# Handle case where state_dict was saved directly or wrapped
if "model_state_dict" in state_dict:
  model.load_state_dict(state_dict["model_state_dict"])
else:
  model.load_state_dict(state_dict)

In [30]:
model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [31]:
# Run evaluation inference
all_preds = []
all_targets = []
class_names = ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']

print("Running inference on test split...")
with torch.no_grad():
  for inputs, targets in test_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.cpu().numpy())
    all_targets.extend(targets.cpu().numpy())

Running inference on test split...


# results

In [32]:
# Generate & Print Report
print("\n=== TEST CLASSIFICATION REPORT ===")
report = classification_report(all_targets, all_preds, target_names=class_names, zero_division=0)
print(report)

# Save report to outputs
outputs_dir = os.path.join(repo_dir, "outputs")
os.makedirs(outputs_dir, exist_ok=True)
with open(os.path.join(outputs_dir, "classification_report_test.txt"), "w") as f:
  f.write(report)


=== TEST CLASSIFICATION REPORT ===
                    precision    recall  f1-score   support

     Mild Dementia       0.16      0.11      0.13       732
 Moderate Dementia       0.00      0.00      0.00       244
      Non Demented       0.86      0.88      0.87     10126
Very mild Dementia       0.44      0.49      0.46      2135

          accuracy                           0.76     13237
         macro avg       0.36      0.37      0.37     13237
      weighted avg       0.74      0.76      0.75     13237



In [33]:
# Plot & Save Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Test Confusion Matrix')
cm_path = os.path.join(outputs_dir, "confusion_matrix_test.png")
plt.savefig(cm_path)
plt.close()
print(f"\nConfusion matrix saved successfully to: {cm_path}")


Confusion matrix saved successfully to: /content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/confusion_matrix_test.png


# improiving imbalance

In [34]:
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Ensure path and load dataset splits
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"
src_path = os.path.join(repo_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from dataset import make_stratified_splits

print("Loading data splits...")
train_loader, val_loader, test_loader, split_info = make_stratified_splits(
    data_dir="/content/dataset/Data",
    batch_size=32,
    split_mode="subject"
)

Loading data splits...


In [35]:
# 2. Automatically compute Class Weights to handle imbalance
# Count how many samples per class exist in the training set
class_counts = [0, 0, 0, 0] # [Mild, Moderate, Non, Very mild] -> match dataset folder order
for _, targets in train_loader:
  for t in targets:
    class_counts[t.item()] += 1

print(f"Training class distribution: {class_counts}")

# Compute inverse frequency weights: total / (num_classes * count)
total_samples = sum(class_counts)
num_classes = len(class_counts)
class_weights = [total_samples / (num_classes * count) if count > 0 else 1.0 for count in class_counts]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"Computed Class-Weighted Loss Weights: {weights_tensor}")

Training class distribution: [3599, 244, 46177, 9516]
Computed Class-Weighted Loss Weights: tensor([ 4.1356, 61.0000,  0.3223,  1.5641], device='cuda:0')


In [36]:
# 3. Setup Model & Weighted Loss Function
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)

model_path = os.path.join(repo_dir, "outputs/best_model.pt")
if os.path.exists(model_path):
  print("Loading existing best_model.pt weights for fine-tuning...")
  state_dict = torch.load(model_path, map_location=device)
  if "model_state_dict" in state_dict:
    model.load_state_dict(state_dict["model_state_dict"])
  else:
    model.load_state_dict(state_dict)

model.to(device)

# PASS THE WEIGHTS INTO CRITERION
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5) # Low learning rate for fine-tuning

Loading existing best_model.pt weights for fine-tuning...


In [37]:
# 4. Run a Quick 1-Epoch Fine-Tuning Loop
print("\n--- Running 1 Epoch with Class-Weighted Loss ---")
model.train()
running_loss = 0.0

for batch_idx, (inputs, targets) in enumerate(train_loader):
  inputs, targets = inputs.to(device), targets.to(device)

  optimizer.zero_grad()
  outputs = model(inputs)
  loss = criterion(outputs, targets)
  loss.backward()
  optimizer.step()

  running_loss += loss.item()
  if batch_idx % 50 == 0:
    print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

print(f"Epoch 1 complete. Average Weighted Loss: {running_loss / len(train_loader):.4f}")


--- Running 1 Epoch with Class-Weighted Loss ---
Batch 0/1861 | Loss: 0.0025
Batch 50/1861 | Loss: 0.0005
Batch 100/1861 | Loss: 0.0016
Batch 150/1861 | Loss: 0.0524
Batch 200/1861 | Loss: 0.0018
Batch 250/1861 | Loss: 0.0023
Batch 300/1861 | Loss: 0.0002
Batch 350/1861 | Loss: 0.0154
Batch 400/1861 | Loss: 0.0008
Batch 450/1861 | Loss: 0.0015
Batch 500/1861 | Loss: 0.0015
Batch 550/1861 | Loss: 0.0011
Batch 600/1861 | Loss: 0.0038
Batch 650/1861 | Loss: 0.0051
Batch 700/1861 | Loss: 0.0010
Batch 750/1861 | Loss: 0.0015
Batch 800/1861 | Loss: 0.0083
Batch 850/1861 | Loss: 0.0054
Batch 900/1861 | Loss: 0.0128
Batch 950/1861 | Loss: 0.0002
Batch 1000/1861 | Loss: 0.0006
Batch 1050/1861 | Loss: 0.0087
Batch 1100/1861 | Loss: 0.0032
Batch 1150/1861 | Loss: 0.0072
Batch 1200/1861 | Loss: 0.0022
Batch 1250/1861 | Loss: 0.0005
Batch 1300/1861 | Loss: 0.0016
Batch 1350/1861 | Loss: 0.0003
Batch 1400/1861 | Loss: 0.0009
Batch 1450/1861 | Loss: 0.0003
Batch 1500/1861 | Loss: 0.0018
Batch 1550/1

In [38]:
# 5. Evaluate the 1-epoch shifted model on the Test Set
model.eval()
all_preds = []
all_targets = []
class_names = ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']

print("\nEvaluating updated model on test split...")
with torch.no_grad():
  for inputs, targets in test_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.cpu().numpy())
    all_targets.extend(targets.cpu().numpy())

print("\n=== NEW TEST REPORT (AFTER 1 EPOCH WEIGHTED LOSS) ===")
print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))


Evaluating updated model on test split...

=== NEW TEST REPORT (AFTER 1 EPOCH WEIGHTED LOSS) ===
                    precision    recall  f1-score   support

     Mild Dementia       0.09      0.09      0.09       732
 Moderate Dementia       0.00      0.00      0.00       244
      Non Demented       0.85      0.85      0.85     10126
Very mild Dementia       0.39      0.43      0.41      2135

          accuracy                           0.73     13237
         macro avg       0.33      0.34      0.34     13237
      weighted avg       0.72      0.73      0.72     13237

